# Ajuste de hiperparámetros de un árbol de decisión

Contenido:

1. ¿Qué son los parámetros e hiper-parámetros?
2. ¿Qué busca el ajuste de hiper-parámetros?
3. Algoritmo GridSearch para el ajuste de hiper-parámetros
4. Lectura y pre-procesamiento del set de datos
5. Entrenamiento y validación de un árbol de regresión básico
6. Ajuste de hiper-parámetros del árbol de regresión
7. Selección y prueba del mejor modelo

## 1. ¿Qué son los parámetros e hiper-parámetros?

> **Parámetros:** son los coeficientes numéricos que se obtienen **automáticamente** con el algoritmo de entrenamiento

> **Hiper-parámetros**: son los coeficientes numéricos o las variables que debemos definir **manualmente** antes de entrenar el árbol

![](parametros-hiperparametros-arbol-de-decision.png)

## 2. ¿Qué busca ajuste de hiper-parámetros?

- Todos los modelos de Machine Learning, en mayor o menor grado, tienden a tener "overfitting"
- El ajuste de hiper-parámetros busca encontrar el conjunto de hiper-parámetros que genere el **menor *overfitting* posible**
- Este ajuste de hiper-parámetros también se conoce como **afinación de hiper-parámetros**

## 3. Algoritmos para el ajuste de hiper-parámetros

Los algoritmos más usados son:

### 3.1. Algoritmo GridSearch

![](algoritmo_grid_search.png)

### 3.2. Algoritmo RandomSearch

Similar al algoritmo GridSearch exceptuando que en el paso 2 **se selecciona aleatoriamente una porción de todas las posibles combinaciones**

En la práctica, siempre que sea posible, se sugiere hacer uso del algoritmo GridSearch.

## 4. Lectura, exploración y pre-procesamiento del set de datos

En este ejercicio usaremos exactamente el mismo set de datos de la lección 8 (cuando construimos un árbol de regresión). Así que acá simplemente se replicará el código visto en dicha lección:

In [1]:
# Leer dataset
import pandas as pd

RUTA = '/Users/miguel/Library/CloudStorage/GoogleDrive-miguel@codificandobits.com/My Drive/02-CODIFICANDOBITS.COM/04-Academia/01-Cursos/28-2024-10-ArbolesDeDecision/data/'
df = pd.read_csv(RUTA + 'alquiler_apartamentos.csv')

# Convertir la variable "price-display" a su representación numérica
df = df[~df['price_display'].str.contains('-')]
df['price'] = df['price_display'].replace({'\$': '', ',': ''}, regex=True).astype(float)

# Eliminar columnas irrelevantes
df.drop(columns=['title', 'body', 'price_display', 'category'], inplace=True)

# Codificación "one-hot" de las variables categóricas
df = pd.get_dummies(df)

# Reordenamiento de las columnas (para dejar "price" como última columna)
df = df[['bathrooms', 'bedrooms', 'square_feet', 'fee_No', 'fee_Yes',
       'pets_allowed_Cats', 'pets_allowed_Cats,Dogs', 'pets_allowed_Dogs',
       'pets_allowed_None', 'price']]
df



,bathrooms,bedrooms,square_feet,fee_No,fee_Yes,pets_allowed_Cats,"pets_allowed_Cats,Dogs",pets_allowed_Dogs,pets_allowed_None,price
0,1.0,1.0,542,1,0,1,0,0,0,2195.0
1,1.5,3.0,1500,1,0,0,1,0,0,1250.0
2,2.0,3.0,1650,1,0,0,0,0,1,1395.0
3,1.0,2.0,820,1,0,0,1,0,0,1600.0
4,1.0,1.0,624,1,0,0,1,0,0,975.0
...,...,...,...,...,...,...,...,...,...,...
44036,1.0,1.0,625,1,0,0,1,0,0,685.0
44037,1.0,1.0,650,1,0,0,1,0,0,798.0
44038,2.0,2.0,921,1,0,0,1,0,0,813.0
44039,1.0,1.0,650,1,0,0,1,0,0,1325.0


Ahora sí podemos crear los sets de datos para entrenamiento/validación y para prueba. Para ello usaremos "train_test_split" de Scikit-Learn para generar las dos particiones (el set de prueba contendrá el 25%) de los datos:

In [2]:
from sklearn.model_selection import train_test_split

# Arreglos x y y
x = df.iloc[:,:-1]
y = df.iloc[:,-1]

# Creación sets de entrenamiento/validación y prueba
x_tr, x_ts, y_tr, y_ts = train_test_split(x, y, 
                                          test_size=0.25, # Tamaño del set de prueba: 25% del total 
                                          random_state=123, # Para reproducibilidad
                                          )

# Imprimir en pantalla los tamaños de los sets que acabamos de crear
print('Tamaño set de entrenamiento/validación: ', x_tr.shape, y_tr.shape)
print('Tamaño set de prueba: ', x_ts.shape, y_ts.shape)

Tamaño set de entrenamiento/validación:  (33009, 9) (33009,)
Tamaño set de prueba:  (11004, 9) (11004,)


## 5. Entrenamiento y validación de un árbol de regresión básico

Antes de hacer el ajuste de hiper-parámetros construiremos un modelo básico que intencionalmente tendrá *overfitting*. Con esto tendremos una referencia para comparar el desempeño del árbol una vez lo hayamos afinado.

Usaremos el RMSE como métrica de desempeño:

In [3]:
# Entrenamiento básico del regresor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import root_mean_squared_error

# Crear instancia del clasificador y entrenar con el set de entrenamiento/validación (x_trvl, y_trvl)
arbol = DecisionTreeRegressor()
arbol.fit(x_tr, y_tr)

# RMSE entrenamiento y prueba

# Entrenamiento
y_pred = arbol.predict(x_tr)
rmse = root_mean_squared_error(y_tr, y_pred)
print("RMSE entrenamiento: ", rmse)

# Prueba
y_pred = arbol.predict(x_ts)
rmse = root_mean_squared_error(y_ts, y_pred)
print("RMSE prueba: ", rmse)

RMSE entrenamiento:  516.2799602240177
RMSE prueba:  767.2804225918668


Perfecto, ya tenemos nuestro árbol de referencia el cual efectivamente tiene "overfitting". Veamos ahora cómo realizar la afinación de hiper-parámetros.

## 6. Ajuste de hiper-parámetros del árbol de regresión

El primer paso es definir los posibles valores para los hiper-parámetros. Para tener una estimación adecuada de estos valores sugiero determinar los parámetros que tiene el árbol sobre-ajustado que acabamos de construir:

In [4]:
print('Profundidad: ', arbol.get_depth())
print('Número de hojas: ', arbol.get_n_leaves())
print('Parámetros: ', arbol.get_params())

Profundidad:  37
Número de hojas:  6796
Parámetros:  {'ccp_alpha': 0.0, 'criterion': 'squared_error', 'max_depth': None, 'max_features': None, 'max_leaf_nodes': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'random_state': None, 'splitter': 'best'}


Y podemos usar estos valores de referencia para definir los posibles valores de los hiper-parámetros:

In [5]:
import numpy as np

hparametros = {'criterion':['squared_error','absolute_error'],
               'max_depth':np.arange(5,21,5).tolist(), # 5, 10, 15, 20
               'min_samples_split': np.arange(3,8,2).tolist(), # 3, 5, 7
               'max_leaf_nodes':np.arange(100,501,100).tolist() # 100, 200, 300, 400, 500
               }
hparametros

{'criterion': ['squared_error', 'absolute_error'],
 'max_depth': [5, 10, 15, 20],
 'min_samples_split': [3, 5, 7],
 'max_leaf_nodes': [100, 200, 300, 400, 500]}

Y calculemos el número total de posibles combinaciones que tendremos, simplemente haciendo el producto entre la cantidad de valores de cada hiper-parámetro:

In [6]:
nro_combs = 1
for key, value in hparametros.items():
    nro_combs *= len(value)
print(nro_combs)

120


Se debe tener en cuenta que entre más combinaciones tengamos posiblemente lograremos tener una mejor afinación pero se incrementará el tiempo de cómputo de la misma.

El segundo paso es crear una instancia de GridSearchCV que con muy pocas líneas de código nos permitirá implementar los pasos 2, 3 y 4 del algoritmo:

2. Generar todas las posibles combinaciones de hiper-parámetros
3. Por cada combinación entrenar y validar un modelo usando el enfoque de **validación cruzada (k-fold cross-validation)**
4. Escoger el mejor set de hiper-parámetros y el mejor modelo

Para crear la instancia debemos definir estos argumentos:

- El tipo de modelo a usar. En este caso será "DecisionTreeRegressor()"
- "param_grid": el listado de hiper-parámetros definidos anteriormente
- "cv": el valor de *k* para llevar a cabo la validación cruzada de cada modelo
- "n_jobs": en ocasiones (cuando tenemos muchos datos y modelos muy grandes) resulta útil aprovechar el uso de múltiples CPUs en caso de que estén disponibles, con el fin de agilizar el cómputo. Para usar todas las CPUs disponibles definiremos n_jobs=-1
- "verbose" (opcional): para que nos imprima en pantalla cada iteración del algoritmo. Usaremos "verbose = 1" para que nos muestre información general de la afinación.

Creemos entonces esta instancia:

In [7]:
from sklearn.model_selection import GridSearchCV

gs = GridSearchCV(DecisionTreeRegressor(random_state=123), param_grid = hparametros, cv = 5, n_jobs=-1, verbose=1)

Y lo único que nos resta es ejecutar el algoritmo para realizar la afinación. En este caso simplemente usamos el método "fit" y le presentamos el set de entrenamiento. La ejecución del algoritmo tomará varios segundos (dependiendo del tipo de procesador que tengamos) pues se tendrán que entrenar múltiples modelos cada uno con diferentes hiper-parámetros:

In [8]:
gs.fit(x_tr,y_tr)

Fitting 5 folds for each of 120 candidates, totalling 600 fits


GridSearchCV(cv=5, estimator=DecisionTreeRegressor(random_state=123), n_jobs=-1,
             param_grid={'criterion': ['squared_error', 'absolute_error'],
                         'max_depth': [5, 10, 15, 20],
                         'max_leaf_nodes': [100, 200, 300, 400, 500],
                         'min_samples_split': [3, 5, 7]},
             verbose=1)

¡Y en este punto ya hemos afinado el árbol de regresión!

Lo que nos resta es extraer la mejor combinación de hiper-parámetros así como el mejor modelo para finalmente ponerlo a prueba.

## 7. Selección y prueba del mejor modelo

Para extraer el mejor conjunto de hiper-parámetros podemos usar el método "best_params_":

In [9]:
mejores_hparams = gs.best_params_
mejores_hparams

{'criterion': 'squared_error',
 'max_depth': 5,
 'max_leaf_nodes': 100,
 'min_samples_split': 5}

Y el mejor modelo usamos "best_estimator_":

In [10]:
mejor_arbol = gs.best_estimator_

In [12]:
mejor_arbol.get_n_leaves()

31

Así que sólo nos resta obtener el desempeño de este estimador con los sets de entrenamiento y prueba (usando como métrica el RMSE):

In [13]:
# Entrenamiento
y_pred = mejor_arbol.predict(x_tr)
rmse = root_mean_squared_error(y_tr, y_pred)
print("RMSE entrenamiento: ", rmse)

# Prueba
y_pred = mejor_arbol.predict(x_ts)
rmse = root_mean_squared_error(y_ts, y_pred)
print("RMSE prueba: ", rmse)

RMSE entrenamiento:  665.0804606916569
RMSE prueba:  718.7148797186442


Y comparemos este resultado con el que obtuvimos antes para el árbol base (sin afinación):

In [14]:
# Entrenamiento
y_pred = arbol.predict(x_tr)
rmse = root_mean_squared_error(y_tr, y_pred)
print("RMSE entrenamiento: ", rmse)

# Prueba
y_pred = arbol.predict(x_ts)
rmse = root_mean_squared_error(y_ts, y_pred)
print("RMSE prueba: ", rmse)

RMSE entrenamiento:  516.2799602240177
RMSE prueba:  767.2804225918668


Efectivamente hemos logrado mejorar el desempeño del árbol:

1. El RMSE del set de entrenamiento empeoró ligeramente (pasando de 516.3 a 665.1 dólares)
2. Pero gracias a esto se mejoró el desempeño con el set de prueba (pasando de 769.2 a 718.7)

Sin embargo, estos resultados son susceptibles de mejora. Se puede incrementar el tamaño de la grilla de búsqueda para de esta forma probar más combinaciones de hiper-parámetros.